In [ ]:
mport pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm

### Polygenic Risk scores to predict between AD and control cases

In [ ]:
# PRS - beta value derived from from Kunkle et al dataset 

import os
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import statsmodels.api as sm


META_RAW   = "/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/BDR_AD_control_PCs_PRS.raw"
EIGENVEC   = "/scratch/c.c2029098/dementia_ml_project/results/PCA/BDR_AD_control.eigenvec"
K_PCS_ADJ  = 7

N_REPS     = 100
TEST_SIZE  = 0.2
N_FOLDS    = 10
OUT_DIR    = "/scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/logistic_regression"

os.makedirs(OUT_DIR, exist_ok=True)


def load_eigenvec(path):
    ev = pd.read_csv(path, sep=r"\s+", engine="python", header=None)
    m = ev.shape[1] - 2
    ev.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, m + 1)]
    pc_cols = [f"PC{i}" for i in range(1, min(K_PCS_ADJ, m) + 1)]
    return ev[["FID", "IID"] + pc_cols].copy(), pc_cols


def run_prs_only_with_posthoc(df_all, ev_df, pc_cols):

    y_all = df_all["PHENOTYPE"].replace({1: 0, 2: 1}).astype(int).values
    X_all = df_all[["PRS"]].copy()

    rng = np.random.default_rng(42)

    cv_aucs, test_aucs, test_aucs_pcadj = [], [], []

    # accumulators to average across reps
    accs, precs, recalls, specs, f1s = [], [], [], [], []
    tns, fps, fns, tps = [], [], [], []

    for rep in range(N_REPS):
        seed = int(rng.integers(0, 2**31 - 1))

        idx = np.arange(len(df_all))
        tr_idx, te_idx, y_tr, y_te = train_test_split(
            idx, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=seed
        )
        X_tr, X_te = X_all.iloc[tr_idx], X_all.iloc[te_idx]

        # CV AUC on training
        kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        fold_aucs = []
        for tr_i, val_i in kf.split(X_tr, y_tr):
            lr_cv = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
            lr_cv.fit(X_tr.iloc[tr_i], y_tr[tr_i])
            p_val = lr_cv.predict_proba(X_tr.iloc[val_i])[:, 1]
            fold_aucs.append(roc_auc_score(y_tr[val_i], p_val))
        cv_aucs.append(float(np.mean(fold_aucs)))

        # Train on full training, evaluate on test
        lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
        lr.fit(X_tr, y_tr)
        p_test = lr.predict_proba(X_te)[:, 1]
        test_auc = roc_auc_score(y_te, p_test)
        test_aucs.append(test_auc)

        # Post-hoc PC adjustment (test set only)
        test_ids = df_all.iloc[te_idx][["FID", "IID"]]
        PCs_te = test_ids.merge(ev_df, on=["FID", "IID"], how="left")[pc_cols].to_numpy()

        adj = LinearRegression()
        adj.fit(PCs_te, p_test)
        p_test_adj = p_test - adj.predict(PCs_te)
        test_aucs_pcadj.append(roc_auc_score(y_te, p_test_adj))

        # Confusion-matrix metrics (threshold=0.5) 
        y_pred = (p_test >= 0.5).astype(int)
        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec  = recall_score(y_te, y_pred)
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1   = f1_score(y_te, y_pred)

        tns.append(tn); fps.append(fp); fns.append(fn); tps.append(tp)
        accs.append(acc); precs.append(prec); recalls.append(rec); specs.append(spec); f1s.append(f1)

        if (rep + 1) % 50 == 0:
            print(f"[PRS_only] {rep+1}/{N_REPS} reps")

    # Averages across reps
    metrics_summary = pd.DataFrame([{
        "cv_auc_mean":               float(np.mean(cv_aucs)),
        "test_auc_mean":             float(np.mean(test_aucs)),
        "test_auc_pc_adjusted_mean": float(np.mean(test_aucs_pcadj)),
        "TN_mean": float(np.mean(tns)), "FP_mean": float(np.mean(fps)),
        "FN_mean": float(np.mean(fns)), "TP_mean": float(np.mean(tps)),
        "accuracy_mean": float(np.mean(accs)),
        "precision_mean": float(np.mean(precs)),
        "recall_mean": float(np.mean(recalls)),
        "specificity_mean": float(np.mean(specs)),
        "f1_mean": float(np.mean(f1s)),
    }])

    return (np.array(cv_aucs), np.array(test_aucs), np.array(test_aucs_pcadj),
            metrics_summary)


def main():
    df = pd.read_csv(META_RAW, sep=r"\s+")
    ev_df, pc_cols = load_eigenvec(EIGENVEC)

    cv, te, te_adj, metrics_summary = run_prs_only_with_posthoc(df, ev_df, pc_cols)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save AUC vectors 
    pd.DataFrame({"rep": np.arange(1, len(cv) + 1), "cv_auc": cv}).to_csv(
        os.path.join(OUT_DIR, f"PRS_only_cv_aucs_{stamp}.tsv"), sep="\t", index=False
    )
    pd.DataFrame({"rep": np.arange(1, len(te) + 1), "test_auc": te}).to_csv(
        os.path.join(OUT_DIR, f"PRS_only_test_aucs_{stamp}.tsv"), sep="\t", index=False
    )
    pd.DataFrame({"rep": np.arange(1, len(te_adj) + 1), "test_auc_pc_adjusted": te_adj}).to_csv(
        os.path.join(OUT_DIR, f"PRS_only_test_aucs_pc_adjusted_{stamp}.tsv"), sep="\t", index=False
    )

    # Save the averages across reps 
    metrics_path = os.path.join(OUT_DIR, f"PRS_only_metrics_summary_{stamp}.tsv")
    metrics_summary.to_csv(metrics_path, sep="\t", index=False)

    print("\n Summary (means across reps)")
    print(f"CV AUC           : mean={cv.mean():.4f}")
    print(f"Test AUC         : mean={te.mean():.4f}")
    print(f"Test AUC (PC-adj): mean={te_adj.mean():.4f}   [K={len(pc_cols)} PCs]")
    print(f"Saved metrics summary to: {metrics_path}")

   
    df_ids = df[["FID", "IID", "PRS", "PHENOTYPE"]].merge(ev_df, on=["FID", "IID"], how="left")
    y_all = df_ids["PHENOTYPE"].replace({1: 0, 2: 1}).astype(int)

main()



[PRS_only] 50/100 reps
[PRS_only] 100/100 reps

 Summary (means across reps)
CV AUC           : mean=0.7784
Test AUC         : mean=0.7855
Test AUC (PC-adj): mean=0.7874   [K=7 PCs]
Saved metrics summary to: /scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/logistic_regression/PRS_only_metrics_summary_20250909_190110.tsv


In [ ]:
# PRS - Beta value from the training dataset 

import os, numpy as np, pandas as pd
from datetime import datetime

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from scipy.special import expit

RAW_FILE        = "/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/BDR_AD_control.raw"
EIGENVEC_FILE   = "/scratch/c.c2029098/dementia_ml_project/results/PCA/BDR_AD_control.eigenvec"
USE_FIRST_N_PCS = 7
N_REPS          = 100
TEST_SIZE       = 0.2
N_FOLDS         = 10
OUT_DIR         = ("/scratch/c.c2029098/dementia_ml_project/results/"
                   "machine_learning/AD_control/logistic_regression/"
                   "SNP_trainDerivedPRS_noPenalty")
os.makedirs(OUT_DIR, exist_ok=True)

NON_FEATURES = {"FID","IID","PAT","MAT","SEX","PHENOTYPE","PRS"}
def pc_cols_present(cols): return [c for c in cols if str(c).startswith("PC")]

def load_raw(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+")
    y = df["PHENOTYPE"].replace({1:0, 2:1}).astype(int)
    return df.loc[y.isin([0,1])].reset_index(drop=True)

def load_pcs(eig_path: str, use_first_n: int):
    ev = pd.read_csv(eig_path, sep=r"\s+", engine="python", header=None)
    m = ev.shape[1] - 2
    ev.columns = ["FID","IID"] + [f"PC{i}" for i in range(1, m+1)]
    k = min(use_first_n, m)
    pc_cols = [f"PC{i}" for i in range(1, k+1)]
    return ev[["FID","IID"] + pc_cols], pc_cols

def get_snp_columns(df: pd.DataFrame):
    exclude = set(NON_FEATURES) | set(pc_cols_present(df.columns))
    return [c for c in df.columns if c not in exclude]

def post_hoc_pc_adjust(scores: np.ndarray, pcs_array) -> np.ndarray:
    pcs_array = np.asarray(pcs_array, dtype=float)
    adj = LinearRegression().fit(pcs_array, scores)
    return scores - adj.predict(pcs_array)

def unpenalised_pipeline():
    return Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("logit",  LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000))
    ])

def main():
    df = load_raw(RAW_FILE)
    y_all = df["PHENOTYPE"].replace({1:0, 2:1}).astype(int).to_numpy()
    snp_cols = get_snp_columns(df)
    X_all = df[snp_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy()

    ids = df[["FID","IID"]].astype(str).copy()
    ev_df, pc_keep = load_pcs(EIGENVEC_FILE, USE_FIRST_N_PCS)

    rng = np.random.default_rng(2025)
    cv_aucs, test_aucs_prs, test_aucs_prs_pcadj = [], [], []
    tns, fps, fns, tps = [], [], [], []
    accs, precs, recalls, specs, f1s, auc_probs = [], [], [], [], [], []

    for rep in range(N_REPS):
        seed = int(rng.integers(0, 2**31 - 1))
        tr_idx, te_idx, y_tr, y_te = train_test_split(
            np.arange(len(y_all)), y_all, test_size=TEST_SIZE,
            stratify=y_all, random_state=seed
        )
        X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]

        # CV AUC on training
        kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        fold_aucs = []
        for tr_i, val_i in kf.split(X_tr_raw, y_tr):
            pipe_fold = unpenalised_pipeline()
            pipe_fold.fit(X_tr_raw[tr_i], y_tr[tr_i])
            p_val = pipe_fold.predict_proba(X_tr_raw[val_i])[:, 1]
            fold_aucs.append(roc_auc_score(y_tr[val_i], p_val))
        cv_aucs.append(np.mean(fold_aucs))

        # Fit final model
        pipe = unpenalised_pipeline()
        pipe.fit(X_tr_raw, y_tr)
        scaler, logit = pipe.named_steps["scaler"], pipe.named_steps["logit"]

        beta_scaled = logit.coef_.ravel()
        safe_std = np.where(scaler.scale_ == 0, 1.0, scaler.scale_)
        beta_raw = beta_scaled / safe_std
        intercept_raw = float(logit.intercept_ - (beta_scaled * scaler.mean_ / safe_std).sum())

        # Save betas
        pd.DataFrame({"SNP": snp_cols + ["<INTERCEPT>"],
                      "beta": list(beta_raw) + [intercept_raw]}
        ).to_csv(os.path.join(OUT_DIR, f"train_derived_prs_weights_rep{rep+1}.tsv"),
                 sep="\t", index=False)

        # Test PRS scores
        prs_test_scores = X_te_raw @ beta_raw + intercept_raw
        test_aucs_prs.append(roc_auc_score(y_te, prs_test_scores))

        # PC adjustment
        pcs_te = ids.iloc[te_idx].merge(ev_df, on=["FID","IID"], how="left")[pc_keep].to_numpy()
        prs_test_scores_adj = post_hoc_pc_adjust(prs_test_scores, pcs_te)
        test_aucs_prs_pcadj.append(roc_auc_score(y_te, prs_test_scores_adj))

        # Confusion metrics
        y_prob = expit(prs_test_scores)
        y_pred = (y_prob >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()
        accs.append(accuracy_score(y_te, y_pred))
        precs.append(precision_score(y_te, y_pred, zero_division=0))
        recalls.append(recall_score(y_te, y_pred))
        specs.append(tn / (tn + fp))
        f1s.append(f1_score(y_te, y_pred))
        auc_probs.append(roc_auc_score(y_te, y_prob))
        tns.append(tn); fps.append(fp); fns.append(fn); tps.append(tp)

    # Save summaries
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pd.DataFrame({
        "rep": np.arange(1, N_REPS+1),
        "cv_auc": cv_aucs,
        "test_auc_prs": test_aucs_prs,
        "test_auc_prs_pc_adj": test_aucs_prs_pcadj
    }).to_csv(os.path.join(OUT_DIR, f"snp_trainDerivedPRS_summary_{stamp}.tsv"), sep="\t", index=False)

    metrics_summary = pd.DataFrame([{
        "TN_mean": np.mean(tns), "FP_mean": np.mean(fps),
        "FN_mean": np.mean(fns), "TP_mean": np.mean(tps),
        "accuracy_mean": np.mean(accs),
        "precision_mean": np.mean(precs),
        "recall_mean": np.mean(recalls),
        "specificity_mean": np.mean(specs),
        "f1_mean": np.mean(f1s),
        "auc_prob_mean": np.mean(auc_probs),
    }])
    metrics_summary.to_csv(os.path.join(OUT_DIR, f"snp_trainDerivedPRS_confusionMetrics_summary_{stamp}.tsv"),
                           sep="\t", index=False)

main()


/tmp/ipykernel_32655/2103520202.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  intercept_raw = float(logit.intercept_ - (beta_scaled * scaler.mean_ / safe_std).sum())
/tmp/ipykernel_32655/2103520202.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  intercept_raw = float(logit.intercept_ - (beta_scaled * scaler.mean_ / safe_std).sum())
/tmp/ipykernel_32655/2103520202.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  intercept_raw = float(logit.intercept_

In [18]:
# Simple logit function - to extract p-value for snps 
import pandas as pd
import statsmodels.api as sm

# Load the .raw file
raw_file = "/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/BDR_AD_control.raw"
df = pd.read_csv(raw_file, sep=r"\s+")

# Define metadata and SNP columns
pheno_col = "PHENOTYPE"
metadata_cols = ['FID', 'IID', 'PAT', 'MAT', 'SEX', pheno_col]
snp_cols = [col for col in df.columns if col not in metadata_cols]

# Define X and y
X = sm.add_constant(df[snp_cols])
y = df[pheno_col].map({1: 0, 2: 1}).astype(int)  # ensure binary 0/1

# Fit logistic regression
model = sm.Logit(y, X)
result = model.fit(disp=0)

# Extract summary stats
summary = result.summary2().tables[1].reset_index().rename(columns={"index": "SNP"})

# Ensure column is numeric
summary["P>|z|"] = pd.to_numeric(summary["P>|z|"], errors="coerce")

# Sort all SNPs by lowest p-value
summary_sorted = summary.sort_values("P>|z|", ascending=True)

# Save full sorted summary
summary_sorted.to_csv(
    "/scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/logistic_regression/all_snps_logistic.tsv",
    sep="\t", index=False, float_format="%.3g"
)

# Save top 10 SNPs separately
top10_path = "/scratch/c.c2029098/dementia_ml_project/results/machine_learning/AD_control/logistic_regression/top10_snps_logistic.tsv"
summary_sorted.head(10).to_csv(top10_path, sep="\t", index=False, float_format="%.3g")

# Print all sorted SNPs
print(summary_sorted)


                      SNP     Coef.  Std.Err.         z         P>|z|  \
60   chr19:44908684:T:C_C  2.359576  0.273677  8.621754  6.593618e-18   
31   chr10:11678309:A:G_G  0.503731  0.188391  2.673858  7.498417e-03   
3      chr2:9558882:A:G_G -0.632685  0.268683 -2.354760  1.853467e-02   
12    chr5:86927378:T:C_C  0.529455  0.236034  2.243129  2.488847e-02   
71   chr21:26784537:C:A_A -0.355144  0.198061 -1.793102  7.295663e-02   
..                    ...       ...       ...       ...           ...   
64   chr19:51225221:C:T_T -0.008505  0.203747 -0.041743  9.667039e-01   
7    chr3:155069722:G:A_A -0.014897  0.374397 -0.039790  9.682606e-01   
66   chr19:54313903:C:A_A -0.008139  0.205893 -0.039528  9.684693e-01   
41  chr14:106665591:G:A_A -0.008321  0.265883 -0.031294  9.750350e-01   
21     chr7:8204382:T:C_T -0.000576  0.282480 -0.002039  9.983733e-01   

      [0.025    0.975]  
60  1.823179  2.895973  
31  0.134491  0.872971  
3  -1.159295 -0.106075  
12  0.066837  0.992074 